# 02 - Data Preprocessing

## Predicting Credit Card Fraud

### Objectives

- Separate features and target variable.
- Exclude non-predictive identifiers.
- Create stratified training and validation sets.
- Encode categorical variables.
- Scale numerical variables when necessary.
- Build preprocessing pipelines.
- Prevent data leakage.

### Target variable

`is_fraud`

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

import matplotlib.pyplot as plt
import seaborn as sns

print("Scikit-learn version:", sklearn.__version__)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

Scikit-learn version: 1.9.1


In [2]:
# Detect project data directory
possible_paths = [
    Path.cwd() / "data" / "raw",
    Path.cwd().parent / "data" / "raw"
]

DATA_DIR = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate data/raw directory"
    )

print("Data directory:", DATA_DIR.resolve())

# Load datasets
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("\nTrain:", train.shape)
print("Test:", test.shape)
print("Submission:", sample_submission.shape)

Data directory: C:\Users\afern\Desktop\Proyectos Personales\Kaggle - Credit Card Fraud\data\raw

Train: (699960, 26)
Test: (299984, 25)
Submission: (299984, 2)


In [3]:
TARGET = "is_fraud"
ID_COL = "id"

# Validate target
assert TARGET in train.columns
assert TARGET not in test.columns

assert train[TARGET].notna().all()

target_values = set(train[TARGET].unique())

assert target_values == {0.0, 1.0}, (
    f"Unexpected target values: {target_values}"
)

# Preserve IDs
train_ids = train[ID_COL].copy()
test_ids = test[ID_COL].copy()

# Separate features and target
X = train.drop(columns=[TARGET, ID_COL]).copy()

y = train[TARGET].astype("int8").copy()

# Test features
X_test = test.drop(columns=[ID_COL]).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

print("\nTarget data type:", y.dtype)

print("\nTarget distribution:")
display(y.value_counts(normalize=True) * 100)

X shape: (699960, 24)
y shape: (699960,)
X_test shape: (299984, 24)

Target data type: int8

Target distribution:


is_fraud
0   98.3300
1    1.6700
Name: proportion, dtype: float64

In [4]:
# Compare feature columns
train_features = set(X.columns)
test_features = set(X_test.columns)

print("Columns missing in test:")
print(train_features - test_features)

print("\nExtra columns in test:")
print(test_features - train_features)

assert train_features == test_features, (
    "Train and test features do not match"
)

# Ensure identical column order
X_test = X_test[X.columns]

print("\nTrain and test columns are correctly aligned.")

Columns missing in test:
set()

Extra columns in test:
set()

Train and test columns are correctly aligned.


In [5]:
# Boolean features
boolean_cols = [
    col for col in X.columns
    if pd.api.types.is_bool_dtype(X[col])
]

# Categorical features
categorical_cols = [
    col for col in X.columns
    if (
        pd.api.types.is_string_dtype(X[col])
        or pd.api.types.is_object_dtype(X[col])
        or isinstance(X[col].dtype, pd.CategoricalDtype)
    )
]

# Numerical features (excluding booleans)
numeric_cols = [
    col for col in X.columns
    if (
        pd.api.types.is_numeric_dtype(X[col])
        and not pd.api.types.is_bool_dtype(X[col])
    )
]

print("Numerical features:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical features:", len(categorical_cols))
print(categorical_cols)

print("\nBoolean features:", len(boolean_cols))
print(boolean_cols)

# Verify all features have been classified
classified_cols = (
    set(numeric_cols)
    | set(categorical_cols)
    | set(boolean_cols)
)

unclassified_cols = set(X.columns) - classified_cols

print("\nUnclassified features:", unclassified_cols)

assert len(unclassified_cols) == 0

Numerical features: 13
['amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes']

Categorical features: 5
['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']

Boolean features: 6
['is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt']

Unclassified features: set()


In [6]:
missing_summary = pd.DataFrame({
    "train_missing": X.isnull().sum(),
    "test_missing": X_test.isnull().sum()
})

display(missing_summary)

assert X.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0

print("No missing values detected.")

,train_missing,test_missing
amount_usd,0,0
hours_since_last_txn,0,0
txn_count_last_24h,0,0
distance_from_home_km,0,0
card_age_months,0,0
customer_age,0,0
account_balance_usd,0,0
cvv_retry_count,0,0
velocity_score,0,0
time_of_day_hour,0,0


No missing values detected.


In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

print("\ny_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (559968, 24)
X_valid: (139992, 24)

y_train: (559968,)
y_valid: (139992,)


In [9]:
split_summary = pd.DataFrame({
    "rows": [
        len(y),
        len(y_train),
        len(y_valid)
    ],
    "fraud_count": [
        int(y.sum()),
        int(y_train.sum()),
        int(y_valid.sum())
    ],
    "fraud_percentage": [
        y.mean() * 100,
        y_train.mean() * 100,
        y_valid.mean() * 100
    ]
}, index=[
    "Complete dataset",
    "Training set",
    "Validation set"
])

display(split_summary)

print(
    "Shared indices:",
    len(set(X_train.index) & set(X_valid.index))
)

assert set(X_train.index).isdisjoint(X_valid.index)

print("Training and validation sets are independent.")

,rows,fraud_count,fraud_percentage
Complete dataset,699960,11689,1.6700
Training set,559968,9351,1.6699
Validation set,139992,2338,1.6701


Shared indices: 0
Training and validation sets are independent.


In [10]:
def create_preprocessor(scale_numeric=True):

    # Numerical transformation
    if scale_numeric:
        numerical_transformer = RobustScaler()
    else:
        numerical_transformer = "passthrough"

    # One-Hot Encoder
    try:
        categorical_transformer = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32
        )

    except TypeError:
        # Compatibility with older Scikit-learn versions
        categorical_transformer = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32
        )

    # Combined preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numerical_transformer,
                numeric_cols
            ),
            (
                "cat",
                categorical_transformer,
                categorical_cols
            ),
            (
                "bool",
                "passthrough",
                boolean_cols
            )
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    return preprocessor

In [11]:
# For linear and distance-based models
preprocessor_scaled = create_preprocessor(
    scale_numeric=True
)

# For tree-based models
preprocessor_tree = create_preprocessor(
    scale_numeric=False
)

print("Scaled preprocessor created.")
print("Tree-based preprocessor created.")

Scaled preprocessor created.
Tree-based preprocessor created.


In [12]:
# Fit only on training data
X_train_processed = preprocessor_scaled.fit_transform(
    X_train
)

# Transform validation data
X_valid_processed = preprocessor_scaled.transform(
    X_valid
)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_valid_processed.shape)

print("\nOutput data type:", X_train_processed.dtype)

Processed training shape: (559968, 54)
Processed validation shape: (139992, 54)

Output data type: float64


In [13]:
processed_feature_names = (
    preprocessor_scaled.get_feature_names_out()
)

print("Original features:", X.shape[1])
print("Processed features:", len(processed_feature_names))

print("\nProcessed feature names:")

display(
    pd.DataFrame({
        "feature_name": processed_feature_names
    })
)

Original features: 24
Processed features: 54

Processed feature names:


,feature_name
0,amount_usd
1,hours_since_last_txn
2,txn_count_last_24h
3,distance_from_home_km
4,card_age_months
5,customer_age
6,account_balance_usd
7,cvv_retry_count
8,velocity_score
9,time_of_day_hour


In [14]:
processed_preview = pd.DataFrame(
    X_train_processed[:5],
    columns=processed_feature_names
)

display(processed_preview)

,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,merchant_risk_score,prior_disputes,merchant_category_Crypto Exchange,merchant_category_Electronics,merchant_category_Fuel,merchant_category_Gaming,merchant_category_Gift Cards,merchant_category_Groceries,merchant_category_Healthcare,merchant_category_Online Retail,merchant_category_Restaurants,merchant_category_Streaming,merchant_category_Travel,merchant_category_Utilities,card_type_Amex,card_type_Discover,card_type_Mastercard,card_type_RuPay,card_type_Visa,auth_method_3D Secure,auth_method_Biometric,auth_method_No Authentication,auth_method_OTP,auth_method_PIN,channel_ATM,channel_Contactless,channel_In-App,channel_Online,channel_POS,device_type_ATM Machine,device_type_Android Phone,device_type_Mac,device_type_POS Terminal,device_type_Smart Watch,device_type_Tablet,device_type_Windows PC,device_type_iPhone,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt
0,3.0340,1.1615,-0.5000,-0.2972,0.2500,-0.8571,2.2024,0.0000,-1.2740,0.5000,0.5000,1.3558,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,19.0549,-0.5567,0.5000,0.4187,0.5000,0.5000,0.2963,0.0000,0.6849,-0.0833,0.7500,0.3865,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,-0.5042,-0.3314,-1.0000,0.1104,-0.8750,-0.3571,0.0140,0.0000,-0.4589,-0.3333,-0.5000,-0.2761,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000
3,0.2309,-0.1416,-0.5000,-0.2397,-0.5000,1.0000,0.4843,0.0000,-0.6712,0.4167,0.0000,0.7791,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.2550,0.1091,1.0000,1.3521,-1.2500,-0.1786,0.9610,1.0000,0.3082,0.2500,-0.2500,0.5521,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000


In [15]:
unseen_categories = {}

for col in categorical_cols:

    train_categories = set(X_train[col].dropna().unique())
    test_categories = set(X_test[col].dropna().unique())

    unseen = test_categories - train_categories

    unseen_categories[col] = list(unseen)

print("Unseen categories in test:")

display(
    pd.DataFrame({
        "feature": unseen_categories.keys(),
        "unseen_categories": unseen_categories.values()
    })
)

Unseen categories in test:


,feature,unseen_categories
0,merchant_category,[]
1,card_type,[]
2,auth_method,[]
3,channel,[]
4,device_type,[]


In [16]:
del X_train_processed
del X_valid_processed
del processed_preview

print("Temporary preprocessing matrices deleted.")

Temporary preprocessing matrices deleted.


# Preprocessing Decisions

## 1. Feature Selection

- Excluded `id` from predictive features.
- Preserved `id` separately for final predictions.
- Separated `is_fraud` as the target variable.

## 2. Data Splitting

- Training set: 80%.
- Validation set: 20%.
- Stratified split to preserve class distribution.
- Random state: 42.

## 3. Missing Values

- No missing values detected.
- No imputation required.

## 4. Numerical Features

- RobustScaler selected for algorithms sensitive to feature scaling.
- Numerical features remain unscaled for tree-based algorithms.
- No logarithmic transformations applied yet.

## 5. Categorical Features

- One-Hot Encoding applied.
- `handle_unknown="ignore"` to handle unseen categories.

## 6. Boolean Features

- Preserved as binary variables.

## 7. Class Imbalance

- No resampling or undersampling applied.
- Original class distribution preserved in training and validation sets.
- Imbalance strategies will be evaluated in a later phase.

## 8. Leakage Prevention

- Preprocessors are fitted exclusively on training data.
- Validation data is only transformed.
- Future cross-validation experiments will use complete Scikit-learn pipelines.

## 9. Pending Investigations

- Evaluate cyclical encoding for `time_of_day_hour` and `day_of_week`.
- Investigate possible transformations for highly skewed numerical variables.
- Investigate the availability and timing of `is_ai_generated_scam_attempt`.